# Fingerprint improvement sweep

Tests the three changes proposed in **`FINGERPRINT_IMPROVE.md`**, which exist because
`RESULTS.md` showed the SNN fingerprint is complementary to ECAPA (pair-score correlation
only +0.22) but far too weak for that to be worth anything (EER 0.345 alone; ~1% when fused).
The bottleneck is the fingerprint, not the fusion — so this sweep attacks the fingerprint.

| variant | epochs | R_EXC | vth_rest | question |
|---|---|---|---|---|
| `baseline` | 16 | 11 | 0.60 | reference |
| `ep4` | 4 | 11 | 0.60 | is 4 epochs as good as 16? (learning is done by epoch 3) |
| `ep4_r5` | 4 | 5 | 0.60 | does a narrower receptive field raise the effective rank? |
| `ep4_r5_vth` | 4 | 5 | 0.45 | does a lower threshold revive the 19% dead hidden units? |

Every variant runs over the **same clips**, so all comparisons are paired.

**The headline metric is `PC1-5`, not just EER.** The core diagnosis is that with
`R_EXC_CHANNEL=11` each hidden neuron pools 23 of 64 channels and neighbours overlap 22/23,
which makes `hidden_activity` *more* redundant (PC1-5 = 0.856) than its own input
(PC1-5 = 0.684). Change #1 targets exactly that number. Lower is better.

**Bonus:** this sweep keeps **several utterances per session**, which the main corpus protocol
does not (it stores exactly one). That makes it the first chance to measure the design's own
claim — that same-session utterances converge to nearly the same fingerprint — and to find out
whether session-level averaging is available as free denoising.

Runs locally (`datasets/vox1`) or on Kaggle; edit `INPUT_ROOT` / `DEV_NN_STYLE` in CONFIG.
The `%%writefile` cells recreate the extractor as real modules so `spawn` workers can import
them (a notebook has no importable `__main__`).

In [ ]:
# ── Environment ──────────────────────────────────────────────────────────────
# On Kaggle, uncomment the installs. Locally the repo venv already has these.
# !pip -q install gammatone || pip -q install git+https://github.com/detly/gammatone.git
# !pip -q install brian2
import shutil
import brian2, gammatone, librosa, soundfile
assert shutil.which("ffmpeg"), "ffmpeg not on PATH — needed to decode .m4a"
print("brian2", brian2.__version__, "| ffmpeg", shutil.which("ffmpeg"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG — the only cell you normally edit
# ══════════════════════════════════════════════════════════════════════════════
import os

# Corpus root. Two supported layouts:
#   DEV_NN_STYLE=True :  {ROOT}/dev_NN/{person}/{session}/{utt}.m4a   (VoxCeleb1 as shipped)
#   DEV_NN_STYLE=False:  {ROOT}/{person}/{session}/{utt}.m4a          (a single shard)
def _find_corpus():
    """Walk up from the working directory looking for datasets/vox1, so the notebook
    works whether it is run from its own folder or from the repo root. On Kaggle this
    finds nothing and falls back to the dataset mount — edit it there."""
    d = os.path.abspath(os.getcwd())
    for _ in range(5):
        c = os.path.join(d, "datasets", "vox1")
        if os.path.isdir(c):
            return c
        d = os.path.dirname(d)
    return "/kaggle/input/datasets/qphulong/vox1-voices"

INPUT_ROOT   = _find_corpus()           # or hard-code a shard path
DEV_NN_STYLE = True
AUDIO_EXTS   = (".m4a", ".wav")

OUT_DIR = "sweep_out"          # one npz per variant lands here

# Sample size. Cost scales linearly with N_WAVS = N_SPEAKERS * SESSIONS * UTTS.
# UTTS_PER_SESSION > 1 is what makes the same-session convergence measurement possible —
# the main corpus protocol stores only one utterance per session, so this sweep is the
# first chance to test that claim. Do not set it to 1.
N_SPEAKERS        = 60
SESSIONS_PER_SPK  = 4
UTTS_PER_SESSION  = 2
SEED              = 1234

WORKERS = os.cpu_count() or 2

# ── The variants being compared (see FINGERPRINT_IMPROVE.md §4) ───────────────
# Every variant runs over the SAME clips, so all comparisons are paired.
VARIANTS = [
    dict(name="baseline",    n_epochs=16, r_exc=11, vth_rest=0.60),
    dict(name="ep4",         n_epochs=4,  r_exc=11, vth_rest=0.60),
    dict(name="ep4_r5",      n_epochs=4,  r_exc=5,  vth_rest=0.60),
    dict(name="ep4_r5_vth",  n_epochs=4,  r_exc=5,  vth_rest=0.45),
]

os.makedirs(OUT_DIR, exist_ok=True)
_n_wavs = N_SPEAKERS * SESSIONS_PER_SPK * UTTS_PER_SESSION
_epoch_units = sum(v["n_epochs"] for v in VARIANTS)
print(f"corpus   : {INPUT_ROOT}  (dev_NN layout = {DEV_NN_STYLE})")
print(f"sample   : {N_SPEAKERS} speakers x {SESSIONS_PER_SPK} sessions x "
      f"{UTTS_PER_SESSION} utts = {_n_wavs} wavs")
print(f"variants : {[v['name'] for v in VARIANTS]}")
print(f"workers  : {WORKERS}")
print(f"rough ETA: {_n_wavs * _epoch_units * 1.5 / max(WORKERS,1) / 60:.0f} min "
      f"(~1.5s per wav per epoch-exposure, measured; plus a one-off ~40s Cython compile)")
print(f"output   : {OUT_DIR}")


### Write the extractor modules to the working dir (imported by spawn workers)

In [ ]:
%%writefile audio_utils.py
import shutil
import subprocess

import numpy as np
import librosa
import soundfile as sf
from gammatone.filters import centre_freqs, make_erb_filters, erb_filterbank


def _ffprobe_sample_rate(path):
    """Native sample rate of `path` via ffprobe, or None if it can't be determined."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe is None:
        return None
    try:
        out = subprocess.run(
            [ffprobe, "-v", "error", "-select_streams", "a:0",
             "-show_entries", "stream=sample_rate",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            check=True, capture_output=True, text=True,
        ).stdout.strip().splitlines()
        return int(out[0]) if out else None
    except (subprocess.CalledProcessError, ValueError):
        return None


def _ffmpeg_decode(path, sr):
    """Decode any ffmpeg-readable container (m4a/aac/mp3/...) to a mono float32
    waveform. If `sr` is None the native rate is probed and preserved; otherwise
    ffmpeg's resampler outputs directly at `sr`."""
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            f"Cannot decode {path!r}: libsndfile failed and ffmpeg is not on PATH. "
            "Install ffmpeg to read m4a/aac audio."
        )
    target_sr = sr if sr is not None else (_ffprobe_sample_rate(path) or 16000)
    proc = subprocess.run(
        [ffmpeg, "-nostdin", "-loglevel", "error", "-i", path,
         "-ac", "1", "-ar", str(target_sr),
         "-f", "f32le", "-acodec", "pcm_f32le", "-"],
        capture_output=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"ffmpeg failed to decode {path!r}: "
            f"{proc.stderr.decode('utf-8', 'ignore').strip()}"
        )
    y = np.frombuffer(proc.stdout, dtype="<f4").astype(np.float32)
    return y, target_sr


def load_audio(path, sr=16000):
    """Load `path` to a mono float32 waveform at `sr` Hz (native rate if `sr` is None).

    Format-robust replacement for ``librosa.load``: wav/flac/ogg are decoded by
    libsndfile (soundfile); m4a/aac and any container libsndfile can't open are
    decoded via ffmpeg. This avoids librosa's audioread m4a fallback, which is
    deprecated and slated for removal in librosa 1.0 (and emits a warning per file).
    """
    try:
        y, sr_native = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(path, sr)
    if y.ndim > 1:                       # multi-channel -> mono (match librosa default)
        y = y.mean(axis=1)
    if sr is not None and sr_native != sr:
        y = librosa.resample(y, orig_sr=sr_native, target_sr=sr)
    return np.ascontiguousarray(y, dtype=np.float32), (sr if sr is not None else sr_native)


def load_mel_spectrogram(
    wav_path: str,
    n_mels: int = 256,
    fmax: int | None = 8000,
    target_frames_per_second: int = 1000,
    normalize: bool = True,
):
    audio, sr = load_audio(wav_path, sr=None)

    hop_length = int(sr / target_frames_per_second)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        fmax=fmax,
        hop_length=hop_length
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    if normalize:
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    return mel_db, sr

def auditory_frontend(
    audio_path,
    sr=16000,
    num_filters=100,
    f_min=50,
    alpha=1.0,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
    eps=1e-6,
):
    """
    Encode an audio waveform into auditory-inspired spike features

    This function implements a biologically inspired auditory pipeline:
    waveform → gammatone filterbank → upstream percentile normalization →
    inner hair cell compression → onset detection → phase signal.

    Normalization is applied **once, upstream** to the (signed) filterbank output,
    before any nonlinearity. Because `log1p(alpha * x)` is not scale-invariant, the
    signal must be brought to a known scale *before* the log so compression is
    consistent across utterances. E, dE and phase are then all derived from the same
    normalized signal — their relative balance is therefore set only by the downstream
    gains, not by independent per-feature normalizations.

    Parameters
    ----------
    audio_path : str
        Path to the input audio file.

    sr : int, default=16000
        Target sampling rate for loading audio.

    num_filters : int, default=100
        Number of ERB-spaced gammatone filters (frequency channels).

    f_min : float, default=50
        Minimum center frequency (Hz) of the filterbank.

    alpha : float, default=1.0
        Compression strength for inner hair cell log compression:
        E = log1p(alpha * |signal_norm|).

    norm_percentile : float, default=99.0
        Percentile of |filterbank output| used as the normalization scale. Robust to
        the loudest transients (top 1% at 99) compared to a plain max.

    clip_val : float, default=1.0
        After dividing by the percentile scale, the normalized signal is clipped to
        [-clip_val, clip_val]. This bounds the input to the log and saturates the
        loudest excursions.

    per_channel : bool, default=False
        If False (default), one global percentile scalar is computed over the whole
        (n_channels, T) magnitude array — this preserves cross-channel relative energy
        (formant/timbre structure useful for speaker discrimination). If True, the
        percentile is computed per channel, equalizing quiet and loud bands.

    eps : float, default=1e-6
        Small constant to avoid division by zero.

    Returns
    -------
    dict
        Dictionary containing encoded auditory representations:

        - "E" : np.ndarray (n_channels, T)
            Log-compressed cochlear energy (IHC output), full-wave rectified.

        - "dE" : np.ndarray (n_channels, T)
            Onset detection signal (half-wave rectified temporal derivative of E).

        - "phase" : np.ndarray (n_channels, T)
            Negative half-wave of the normalized filterbank output. Complementary in
            polarity to E's full-wave energy, so it is not redundant with E.

        - "cf" : np.ndarray (n_channels,)
            Center frequencies of filterbank channels (low → high)

        - "sr" : int
            Sampling rate of processed audio

    Notes
    -----
    Processing pipeline:

    1. Audio loading
    2. ERB-spaced gammatone filterbank
    3. Upstream percentile normalization + clip (on the signed signal)
    4. Inner hair cell log compression (full-wave): E = log1p(alpha * |sig_norm|)
    5. Onset detection via positive temporal derivative of E
    6. Phase signal: negative half-wave of sig_norm

    All channel outputs are ordered from **low → high frequency**.
    """

    # ==============================
    # 1. Load audio
    # ==============================
    signal, sr = load_audio(audio_path, sr=sr)

    # ==============================
    # 2. Gammatone filterbank
    # ==============================
    cf = centre_freqs(sr, num_filters, f_min)
    erb_filters = make_erb_filters(sr, cf)

    filtered_signals = erb_filterbank(signal, erb_filters)

    # reorder HIGH→LOW → LOW→HIGH
    cf = cf[::-1]
    filtered_signals = filtered_signals[::-1]

    signals = filtered_signals
    n_channels, T = signals.shape

    # ==============================
    # 3. Upstream percentile normalization (on the signed signal, before any
    #    nonlinearity). One scale derived from |signals|, then clip. This keeps
    #    the log compression in a consistent regime across utterances and puts
    #    E / dE / phase on a single shared reference frame.
    # ==============================
    if per_channel:
        scale = np.percentile(np.abs(signals), norm_percentile, axis=1, keepdims=True)
    else:
        scale = np.percentile(np.abs(signals), norm_percentile)
    sig_n = np.clip(signals / (scale + eps), -clip_val, clip_val)

    # ==============================
    # 4. Inner Hair Cell Compression (full-wave)
    # ==============================
    E = np.log1p(alpha * np.abs(sig_n))

    # ==============================
    # 5. Onset detection (positive temporal derivative of E)
    # ==============================
    dE = np.diff(E, axis=1, prepend=E[:, :1])
    dE[dE < 0] = 0

    # ==============================
    # 6. Phase signal: negative half-wave of the normalized signal.
    #    Complementary in polarity to E's full-wave energy → not redundant with E.
    # ==============================
    phase_signal = np.maximum(-sig_n, 0)

    return {
        "E": E,
        "dE": dE,
        "phase": phase_signal,
        "cf": cf,
        "sr": sr,
    }


In [ ]:
%%writefile spike_encoding.py
import numpy as np
from audio_utils import auditory_frontend

def compute_spike_input_current(
    audio_path,
    sustained_per_band=5,
    onset_per_band=2,
    phase_per_band=2,
    scale=1,
    sust_gain=1.0,
    onset_gain=2.0,
    phase_gain=1.0,
    sust_spread_min=0.6,
    sust_spread_max=1.4,
    audio_sample_rate=16000,
    simulation_sample_rate=1000,
    num_filters=100,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
):
    """
    Convert an audio file into a downsampled input current matrix for a spiking neural network

    This function takes auditory features produced by `auditory_frontend()` and expands
    them into multiple neuron types per cochlear frequency band. Each neuron type
    represents different auditory response characteristics inspired by biological
    auditory nerve fibers.

    Pipeline
    --------
    1. Audio → auditory feature extraction via `auditory_frontend()`
    2. Obtain three feature maps:
        - E     : sustained energy (IHC compressed output)
        - dE    : onset energy (positive temporal derivative)
        - phase : rectified gammatone signal
    3. Generate multiple neurons per frequency band:
        - sustained neurons (energy response, spread across gain multipliers)
        - onset neurons (transient response)
        - phase neurons (phase locking)
    4. Apply gain scaling and small Gaussian noise.
    5. Downsample from `audio_sample_rate` to `simulation_sample_rate` by
       block-averaging across the decimation factor.
    6. Return a time-varying current matrix suitable for driving LIF neurons.

    Parameters
    ----------
    audio_path : str
        Path to the input audio (.wav) file.

    sustained_per_band : int, default=5
        Number of neurons per frequency band that encode sustained energy (E).

    onset_per_band : int, default=2
        Number of neurons per band that encode onset activity (dE).

    phase_per_band : int, default=2
        Number of neurons per band that encode phase-locking signals.

    scale : float, default=1
        Global gain multiplier applied to all input currents.

    sust_gain : float, default=1.3
        Gain factor for sustained-energy neurons.

    onset_gain : float, default=2.0
        Gain factor for onset neurons.

    phase_gain : float, default=1.0
        Gain factor for phase-locking neurons.

    sust_spread_min : float, default=0.6
        Minimum multiplicative factor applied across sustained neurons within a band.

    sust_spread_max : float, default=1.4
        Maximum multiplicative factor applied across sustained neurons within a band.

    audio_sample_rate : int, default=16000
        Sampling rate (Hz) of the raw audio and the auditory feature maps produced
        by `auditory_frontend()`.

    simulation_sample_rate : int, default=1000
        Target sampling rate (Hz) for the Brian2 simulation (i.e. 1 / defaultclock.dt).
        The current matrix is downsampled from `audio_sample_rate` to this rate by
        block-averaging. Must evenly divide `audio_sample_rate`.

    norm_percentile : float, default=99.0
        Percentile used by `auditory_frontend` to normalize the filterbank output
        before the log nonlinearity.

    clip_val : float, default=1.0
        Clip bound applied to the normalized filterbank signal in `auditory_frontend`.

    per_channel : bool, default=False
        If True, `auditory_frontend` normalizes per channel instead of globally.

    Returns
    -------
    I_sim : np.ndarray, shape (N_in, T_sim)
        Simulation-ready input current matrix, where
        T_sim = T // decimation_factor.

    T_sim : int
        Number of time steps after downsampling, corresponding to the
        total simulation duration in Brian2 timesteps.
    """

    feats = auditory_frontend(
        audio_path,
        num_filters=num_filters,
        norm_percentile=norm_percentile,
        clip_val=clip_val,
        per_channel=per_channel,
    )

    E = feats["E"]
    dE = feats["dE"]
    phase = feats["phase"]

    n_channels, T = E.shape

    g_sust = sust_gain
    g_onset = onset_gain
    g_phase = phase_gain

    neurons_per_band = sustained_per_band + onset_per_band + phase_per_band
    N_in = n_channels * neurons_per_band

    I = np.zeros((N_in, T), dtype=np.float32)

    idx = 0
    for ch in range(n_channels):

        spread = np.linspace(sust_spread_min, sust_spread_max, sustained_per_band)
        for mult in spread:
            I[idx] = g_sust * mult * scale * E[ch]
            idx += 1

        for _ in range(onset_per_band):
            I[idx] = g_onset * scale * dE[ch]
            idx += 1

        for _ in range(phase_per_band):
            I[idx] = g_phase * scale * phase[ch]
            idx += 1

    I += 0.01 * np.random.randn(*I.shape).astype(np.float32)
    assert audio_sample_rate % simulation_sample_rate == 0, (
        f"audio_sample_rate ({audio_sample_rate}) must be divisible by "
        f"simulation_sample_rate ({simulation_sample_rate})"
    )
    decimation_factor = audio_sample_rate // simulation_sample_rate
    # Trim to nearest multiple so reshape never fails
    T_trim = (T // decimation_factor) * decimation_factor
    I_sim  = I[:, :T_trim].reshape(N_in, -1, decimation_factor).mean(axis=2)
    T_sim  = I_sim.shape[1]

    return I_sim, T_sim


In [ ]:
%%writefile _fp_sweep_core.py
"""
_fp_sweep_core.py
=================
Variant-parameterised fingerprint extractor for the improvement sweep.

Same network as `ecapa_film_snn/prepare_fingerprints.ipynb`'s `_fingerprint_core.py`
(which mirrors `tonotopic_plasticity_bound/train.py`), with THREE knobs exposed so the
sweep can test the changes proposed in FINGERPRINT_IMPROVE.md:

    n_epochs   — full-clip exposures (baseline 16; the analysis says learning is done by ~3)
    r_exc      — R_EXC_CHANNEL, the tonotopic radius (baseline 11 => each hidden neuron
                 pools 2*11+1 = 23 of 64 channels, so neighbours overlap 22/23)
    vth_rest   — hidden resting threshold (baseline 0.6; lowering it is the cheap test for
                 the 24/128 permanently-dead hidden units)

Everything else is verbatim from the baseline core. Note the L1 column normalisation makes
the r_exc change roughly drive-neutral: the column sum stays NORM_LIMIT_EXC regardless of
fan-in, so fewer inputs simply means each is proportionally stronger.

Per-variant output shape: in/hid weights are (2, 2*r_exc+1, 128) — the tensor SHRINKS as
r_exc shrinks, which is the point (less redundant pooling, fewer parameters).

Import BEFORE numpy in driver code so the BLAS pinning below takes effect.
"""

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import sys
from types import SimpleNamespace

import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
from spike_encoding import compute_spike_input_current

from brian2 import (
    NeuronGroup, Synapses, SpikeMonitor, TimedArray, Network, network_operation,
    defaultclock, prefs, BrianLogger, ms, second,
)

prefs.codegen.target = 'cython'
prefs.codegen.runtime.cython.multiprocess_safe = True
BrianLogger.suppress_name('method_choice')
BrianLogger.suppress_name('unused_brian_object')
prefs.logging.console_log_level = 'ERROR'

# ── Fixed hyperparameters (verbatim from the baseline core) ───────────────────
N_IN = 128
N_H  = 128
DT_SIM  = 1 * ms
CLIP_MS = 2000

tau_m       = 40 * ms
tau_a       = 40 * ms
beta        = 3.5
tau_current = 1 * ms
v_th_in     = 1.0

tau_h    = 150 * ms
tau_vth  = 60 * ms
vth_jump = 1.0
tau_r    = 10 * ms
sigma_noise = 0.03 * second**(-0.5)

taupre  = 20 * ms
taupost = 20 * ms
tau_x   = 100 * ms
tau_y   = 125 * ms
A3PRE_CENTER  =  0.004
A3POST_CENTER = -0.002

wmin         = 0.0
WMAX_CENTER  = 1.0
APRE_CENTER  =  0.008
APOST_CENTER = -0.0096

W_INH_CENTER = 1.0
W_INH_MIN    = 0.0
APRE_INH     = 0.004
APOST_INH    = -0.0048

N_CHANNELS    = 64
N_PER_CHANNEL = N_IN // N_CHANNELS
p_EXC         = 3

NORM_LIMIT_EXC = 1.0455
NORM_LIMIT_INH = 0.40
NORM_DT        = 25 * ms

ENCODER_KWARGS = dict(
    scale=1.0, num_filters=64, sustained_per_band=1, onset_per_band=1, phase_per_band=0,
    sust_gain=0.3, onset_gain=3.0, sust_spread_min=1, sust_spread_max=1,
)


# ═══════════════════════════════════════════════════════════════════════════════
# Per-variant precompute
# ═══════════════════════════════════════════════════════════════════════════════

def make_params(n_epochs=16, r_exc=11, vth_rest=0.6, clip_ms=CLIP_MS, name="variant"):
    """All r_exc-dependent matrices for one variant. Deterministic and picklable-free
    (workers rebuild it themselves from the small config dict)."""
    ch_i = (np.arange(N_IN) // N_PER_CHANNEL).reshape(-1, 1)
    ch_j = (np.arange(N_H) // N_PER_CHANNEL).reshape(1, -1)
    dist = np.abs(ch_i - ch_j)
    dist = np.minimum(dist, N_CHANNELS - dist)

    topo = np.maximum(0.0, 1.0 - (dist / r_exc) ** p_EXC)
    mask_ih = dist <= r_exc
    src_ih, tgt_ih = np.where(mask_ih)

    # Inhibitory Jaccard connectivity, window tied to the same radius.
    ch_h = np.arange(N_H) // N_PER_CHANNEL
    d_hh = np.abs(ch_h.reshape(-1, 1) - ch_h.reshape(1, -1))
    d_hh = np.minimum(d_hh, N_CHANNELS - d_hh)
    window = 2 * r_exc + 1
    overlap = np.maximum(0, window - d_hh)
    jac = np.where(overlap > 0, overlap / (window + d_hh), 0.0)
    mask_hh = (overlap > 0) & (~np.eye(N_H, dtype=bool))
    src_hh, tgt_hh = np.where(mask_hh)

    w_ih_init = np.zeros((N_IN, N_H))
    w_ih_init[src_ih, tgt_ih] = (WMAX_CENTER * topo)[src_ih, tgt_ih]
    for j in range(N_H):
        rows = src_ih[tgt_ih == j]
        s = w_ih_init[rows, j].sum()
        if s > 0:
            w_ih_init[rows, j] *= NORM_LIMIT_EXC / s

    rng = np.random.RandomState(42)
    w_hh_init = np.zeros((N_H, N_H))
    w_hh_init[src_hh, tgt_hh] = rng.uniform(0.01, 0.02, size=src_hh.shape[0])

    offsets = np.arange(-r_exc, r_exc + 1)
    n_off = len(offsets)
    ch_row = np.arange(N_H) // N_PER_CHANNEL
    off_ch = (ch_row[None, :] + offsets[:, None]) % N_CHANNELS
    idx = np.stack([off_ch * N_PER_CHANNEL + t for t in range(N_PER_CHANNEL)])
    shape = (N_PER_CHANNEL, n_off, N_H)

    return SimpleNamespace(
        name=name, n_epochs=n_epochs, r_exc=r_exc, vth_rest=vth_rest, clip_ms=clip_ms,
        src_ih=src_ih, tgt_ih=tgt_ih, src_hh=src_hh, tgt_hh=tgt_hh,
        wmax=WMAX_CENTER * topo, apre=APRE_CENTER * topo, apost=APOST_CENTER * topo,
        a3pre=A3PRE_CENTER * topo, a3post=A3POST_CENTER * topo,
        wmax_inh=W_INH_CENTER * jac, apre_inh=APRE_INH * jac, apost_inh=APOST_INH * jac,
        w_ih_init=w_ih_init, w_hh_init=w_hh_init,
        idx=idx, j_row=np.broadcast_to(np.arange(N_H), shape).copy(), shape=shape,
    )


# ═══════════════════════════════════════════════════════════════════════════════
# Network
# ═══════════════════════════════════════════════════════════════════════════════

def build_network(P):
    defaultclock.dt = DT_SIM
    vth_rest = P.vth_rest

    eqs_in = """
    dv/dt = (-v - a) / tau_m + I_timed(t, i) / tau_current : 1
    da/dt = -a / tau_a : 1
    """
    G_in = NeuronGroup(N_IN, eqs_in, threshold="v > v_th_in",
                       reset="v=0; a+=beta", refractory=2 * ms, method="euler")
    G_in.namespace["I_timed"] = TimedArray(np.zeros((1, N_IN), dtype=float), dt=DT_SIM)

    eqs_h = f"""
    dv/dt       = -v / tau_h + sigma_noise * xi   : 1
    dvth/dt     = -(vth - {vth_rest}) / tau_vth   : 1
    dtrace_r/dt = -trace_r / tau_r                : 1
    """
    G_h = NeuronGroup(N_H, eqs_h, threshold="v > vth",
                      reset=f"v=0; vth=vth+{vth_jump}; trace_r=1;", method="euler")

    stdp_model = """
    w          : 1
    dapre/dt   = -apre  / taupre  : 1 (event-driven)
    dapost/dt  = -apost / taupost : 1 (event-driven)
    dr1/dt     = -r1 / taupre      : 1 (event-driven)
    dr2/dt     = -r2 / tau_x       : 1 (event-driven)
    do1/dt     = -o1 / taupost     : 1 (event-driven)
    do2/dt     = -o2 / tau_y       : 1 (event-driven)
    wmax_syn   : 1
    Apre_syn   : 1
    Apost_syn  : 1
    A3pre_syn  : 1
    A3post_syn : 1
    """
    on_pre = (f"v_post += w * (1 - trace_r_post)\n"
              f"apre += Apre_syn\n"
              f"w = clip(w + apost*(w-{wmin}) + A3post_syn*o1*r2*(w-{wmin}), {wmin}, wmax_syn)\n"
              f"r1 += 1\nr2 += 1")
    on_post = (f"apost += Apost_syn\n"
               f"w = clip(w + apre*(wmax_syn-w) + A3pre_syn*r1*o2*(wmax_syn-w), {wmin}, wmax_syn)\n"
               f"o1 += 1\no2 += 1")

    S_ih = Synapses(G_in, G_h, model=stdp_model, on_pre=on_pre, on_post=on_post)
    S_ih.connect(i=P.src_ih, j=P.tgt_ih)
    s_i, t_i = np.array(S_ih.i), np.array(S_ih.j)
    S_ih.wmax_syn   = P.wmax[s_i, t_i]
    S_ih.Apre_syn   = P.apre[s_i, t_i]
    S_ih.Apost_syn  = P.apost[s_i, t_i]
    S_ih.A3pre_syn  = P.a3pre[s_i, t_i]
    S_ih.A3post_syn = P.a3post[s_i, t_i]

    stdp_inh = """
    w_inh          : 1
    dapre_inh/dt   = -apre_inh  / taupre  : 1 (event-driven)
    dapost_inh/dt  = -apost_inh / taupost : 1 (event-driven)
    wmax_inh_syn   : 1
    Apre_inh_syn   : 1
    Apost_inh_syn  : 1
    """
    on_pre_inh = (f"v_post -= w_inh * (1 - trace_r_post)\n"
                  f"apre_inh += Apre_inh_syn\n"
                  f"w_inh = clip(w_inh + apost_inh*(w_inh-{W_INH_MIN}), {W_INH_MIN}, wmax_inh_syn)")
    on_post_inh = (f"apost_inh += Apost_inh_syn\n"
                   f"w_inh = clip(w_inh + apre_inh*(wmax_inh_syn-w_inh), {W_INH_MIN}, wmax_inh_syn)")

    S_hh = Synapses(G_h, G_h, model=stdp_inh, on_pre=on_pre_inh, on_post=on_post_inh)
    S_hh.connect(i=P.src_hh, j=P.tgt_hh)
    s_h, t_h = np.array(S_hh.i), np.array(S_hh.j)
    S_hh.wmax_inh_syn  = P.wmax_inh[s_h, t_h]
    S_hh.Apre_inh_syn  = P.apre_inh[s_h, t_h]
    S_hh.Apost_inh_syn = P.apost_inh[s_h, t_h]
    S_hh.w_inh         = P.w_hh_init[s_h, t_h]

    spike_in, spike_hid = SpikeMonitor(G_in), SpikeMonitor(G_h)
    wmax_a, wmax_ia = np.array(S_ih.wmax_syn), np.array(S_hh.wmax_inh_syn)

    @network_operation(dt=NORM_DT, when='end')
    def normalize_weights():
        w = np.array(S_ih.w)
        cs = np.bincount(t_i, weights=w, minlength=N_H)
        sc = np.where(cs > NORM_LIMIT_EXC, NORM_LIMIT_EXC / cs, 1.0)
        S_ih.w[:] = np.clip(w * sc[t_i], wmin, wmax_a)
        wi = np.array(S_hh.w_inh)
        csi = np.bincount(t_h, weights=wi, minlength=N_H)
        sci = np.where(csi > NORM_LIMIT_INH, NORM_LIMIT_INH / csi, 1.0)
        S_hh.w_inh[:] = np.clip(wi * sci[t_h], W_INH_MIN, wmax_ia)

    net = Network(G_in, G_h, S_ih, S_hh, spike_in, spike_hid, normalize_weights)
    G_h.vth = vth_rest
    net.store('init')
    return SimpleNamespace(net=net, P=P, G_in=G_in, G_h=G_h, S_ih=S_ih, S_hh=S_hh,
                           src_ih=s_i, tgt_ih=t_i, src_hh=s_h, tgt_hh=t_h,
                           spike_in=spike_in, spike_hid=spike_hid)


def warmup(h):
    h.S_ih.w     = h.P.w_ih_init[h.src_ih, h.tgt_ih]
    h.S_hh.w_inh = h.P.w_hh_init[h.src_hh, h.tgt_hh]
    h.G_in.namespace["I_timed"] = TimedArray(np.zeros((5, N_IN), dtype=float), dt=DT_SIM)
    h.net.run(5 * ms)
    h.net.restore('init')


def train_fingerprint(h, wav_path):
    P = h.P
    try:
        I, T = compute_spike_input_current(wav_path, **ENCODER_KWARGS)
    except Exception as e:
        print(f"  [skip {wav_path}: {e}]")
        return None
    if T > P.clip_ms:
        T = P.clip_ms
        I = I[:, :T]

    h.net.restore('init')
    h.G_in.namespace["I_timed"] = TimedArray(
        np.tile(I, (1, P.n_epochs)).T.astype(float), dt=DT_SIM)
    h.S_ih.w     = P.w_ih_init[h.src_ih, h.tgt_ih]
    h.S_ih.apre  = 0; h.S_ih.apost = 0
    h.S_ih.r1 = 0; h.S_ih.r2 = 0; h.S_ih.o1 = 0; h.S_ih.o2 = 0
    h.S_hh.w_inh = P.w_hh_init[h.src_hh, h.tgt_hh]
    h.S_hh.apre_inh = 0; h.S_hh.apost_inh = 0

    h.net.run(P.n_epochs * T * DT_SIM)

    w_ih = np.zeros((N_IN, N_H), dtype=np.float32)
    w_ih[h.src_ih, h.tgt_ih] = np.array(h.S_ih.w)
    w_hh = np.zeros((N_H, N_H), dtype=np.float32)
    w_hh[h.src_hh, h.tgt_hh] = np.array(h.S_hh.w_inh)

    t0 = (P.n_epochs - 1) * T
    it, ii = np.array(h.spike_in.t / ms), np.array(h.spike_in.i)
    ht, hi = np.array(h.spike_hid.t / ms), np.array(h.spike_hid.i)
    ic = np.bincount(ii[it >= t0], minlength=N_IN).astype(np.int64)
    hc = np.bincount(hi[ht >= t0], minlength=N_H).astype(np.int64)
    return w_ih, w_hh, ic, hc


def _norm_act(c):
    d = np.percentile(c, 99)
    if d <= 0:
        return np.zeros_like(c, dtype=np.float16)
    return np.clip(c / d, 0.0, 1.0).astype(np.float16)


def fingerprint_to_sample(P, w_ih, w_hh, ic, hc):
    isil, hsil = ic == 0, hc == 0
    iw = w_ih[P.idx, P.j_row].astype(np.float32)
    iw[isil[P.idx] | hsil[None, None, :]] = 0.0
    hw = w_hh[P.idx, P.j_row].astype(np.float32)
    hw[hsil[P.idx] | hsil[None, None, :]] = 0.0
    return iw.astype(np.float16), hw.astype(np.float16), _norm_act(ic), _norm_act(hc)


# ── Worker plumbing ───────────────────────────────────────────────────────────
_H = None


def _init_worker(cfg):
    global _H
    _H = build_network(make_params(**cfg))
    warmup(_H)


def process_one(entry):
    pid, sid, fn, lab = entry['person_id'], entry['session_id'], entry['file_name'], entry['label']
    out = train_fingerprint(_H, entry['wav_path'])
    if out is None:
        return (pid, sid, fn, lab, None)
    return (pid, sid, fn, lab, fingerprint_to_sample(_H.P, *out))


In [ ]:
# ── Enumerate wavs: N speakers x SESSIONS each x UTTS per session ─────────────
# Unlike the main pipeline (1 random utterance per session), this KEEPS several
# utterances per session on purpose — that is what allows the within-session vs
# between-session comparison in the evaluation cell.
from pathlib import Path
import numpy as np

rng = np.random.default_rng(SEED)
root = Path(INPUT_ROOT)
assert root.is_dir(), f"INPUT_ROOT not found: {root}"

bases = sorted(d for d in root.glob("dev_*") if d.is_dir()) if DEV_NN_STYLE else [root]
assert bases, f"no dev_* folders under {root} — set DEV_NN_STYLE=False for a flat shard"

people = []
for b in bases:
    people += [d for d in sorted(b.iterdir()) if d.is_dir()]
assert len(people) >= N_SPEAKERS, f"only {len(people)} speakers under {root}"

entries = []
for pdir in rng.permutation(np.array(people, dtype=object)):
    sessions = [d for d in sorted(pdir.iterdir()) if d.is_dir()]
    usable = []
    for sdir in sessions:
        wavs = sorted(f.name for f in sdir.iterdir()
                      if f.is_file() and f.suffix.lower() in AUDIO_EXTS)
        if len(wavs) >= UTTS_PER_SESSION:
            usable.append((sdir, wavs))
    if len(usable) < SESSIONS_PER_SPK:
        continue                       # speaker cannot fill the quota — skip entirely
    for sdir, wavs in usable[:SESSIONS_PER_SPK]:
        pick = rng.choice(len(wavs), UTTS_PER_SESSION, replace=False)
        for w in (wavs[k] for k in pick):
            entries.append(dict(person_id=pdir.name, session_id=sdir.name, file_name=w,
                                label=f"{pdir.name}/{sdir.name}/{w}",
                                wav_path=str(sdir / w)))
    if len({e["person_id"] for e in entries}) >= N_SPEAKERS:
        break

n_spk = len({e["person_id"] for e in entries})
n_ses = len({(e["person_id"], e["session_id"]) for e in entries})
print(f"{len(entries)} wavs | {n_spk} speakers | {n_ses} sessions "
      f"| {len(entries)/max(n_ses,1):.1f} utts per session")
assert n_spk >= N_SPEAKERS, (
    f"only {n_spk} speakers had {SESSIONS_PER_SPK} sessions with >= "
    f"{UTTS_PER_SESSION} utterances each — lower the quotas")


In [ ]:
# ── Run every variant over the SAME clips ────────────────────────────────────
# One npz per variant in OUT_DIR. Idempotent: an existing npz is reused, so a
# crashed or interrupted sweep resumes instead of restarting.
import multiprocessing as mp
import time
import numpy as np
import _fp_sweep_core as C

ctx = mp.get_context("spawn")

for cfg in VARIANTS:
    out_path = os.path.join(OUT_DIR, f"fp_{cfg['name']}.npz")
    if os.path.exists(out_path):
        print(f"[{cfg['name']}] cache hit -> {out_path}")
        continue

    print(f"\n[{cfg['name']}] n_epochs={cfg['n_epochs']} r_exc={cfg['r_exc']} "
          f"vth_rest={cfg['vth_rest']}", flush=True)
    t0 = time.time()
    # Warm the Cython cache in the parent so the workers all reuse it. Equations
    # embed vth_rest as a literal, so each distinct vth_rest compiles once.
    C.warmup(C.build_network(C.make_params(**cfg)))
    print(f"  warm-up {time.time()-t0:.0f}s", flush=True)

    rows, n_skip = [], 0
    t0 = time.time()
    with ctx.Pool(WORKERS, initializer=C._init_worker, initargs=(cfg,)) as pool:
        for k, (pid, sid, fn, lab, payload) in enumerate(
                pool.imap_unordered(C.process_one, entries), 1):
            if payload is None:
                n_skip += 1
            else:
                rows.append((*payload, pid, sid, fn, lab))
            if k % 50 == 0:
                el = time.time() - t0
                print(f"  {k}/{len(entries)}  {el:.0f}s  eta "
                      f"{el/k*(len(entries)-k):.0f}s", flush=True)

    assert rows, f"[{cfg['name']}] produced no fingerprints"
    np.savez(out_path,
             in_weights=np.stack([r[0] for r in rows]).astype(np.float16),
             hid_weights=np.stack([r[1] for r in rows]).astype(np.float16),
             input_activity=np.stack([r[2] for r in rows]).astype(np.float16),
             hidden_activity=np.stack([r[3] for r in rows]).astype(np.float16),
             person_ids=np.array([r[4] for r in rows]),
             session_ids=np.array([r[5] for r in rows]),
             file_names=np.array([r[6] for r in rows]),
             labels=np.array([r[7] for r in rows]),
             sec_per_wav=(time.time() - t0) / max(len(rows), 1),
             **{k: v for k, v in cfg.items() if k != "name"})
    print(f"[{cfg['name']}] {len(rows)} fingerprints ({n_skip} skipped) in "
          f"{time.time()-t0:.0f}s -> {out_path}")


In [ ]:
# ── Evaluate every variant on identical clips ────────────────────────────────
# All metrics are numpy-only. Every variant sees the same wavs, so the comparison
# is paired and the (small) speaker sample affects all rows equally.
import numpy as np

COMPONENTS = ("in_weights", "hid_weights", "input_activity", "hidden_activity")


def _flat(d, keys):
    n = len(d["person_ids"])
    parts = []
    for k in keys:
        a = np.nan_to_num(np.asarray(d[k], dtype=np.float32).reshape(n, -1))
        sd = a.std(0); mu = a.mean(0)
        parts.append((a - mu) / np.maximum(sd, 1e-6))    # z-score per block, then concat
    return np.concatenate(parts, axis=1)


def _pc_frac(X, ks=(5, 20)):
    Xc = X - X.mean(0)
    tot = (Xc ** 2).sum()
    if tot <= 0:
        return [float("nan")] * len(ks)
    s = np.linalg.svd(Xc, full_matrices=False, compute_uv=False)
    ev = np.cumsum(s ** 2) / tot
    return [float(ev[min(k, len(ev)) - 1]) for k in ks]


def _fisher(X, spk):
    cls, inv = np.unique(spk, return_inverse=True)
    S, N, D = len(cls), *X.shape
    mu = X.mean(0)
    cnt = np.bincount(inv, minlength=S).astype(float)
    sums = np.zeros((S, D)); np.add.at(sums, inv, X)
    mu_s = sums / cnt[:, None]
    within = ((X - mu_s[inv]) ** 2).sum(0) / max(N - S, 1)
    between = (cnt[:, None] * (mu_s - mu) ** 2).sum(0) / max(S - 1, 1)
    live = X.std(0) > 1e-8
    return float(np.median((between / np.maximum(within, 1e-12))[live])) if live.any() else np.nan


def _retrieval(X, spk, rec):
    """Session-free all-pairs cosine retrieval — same protocol as the main notebook.
    Exact EER by sorting every valid pair."""
    Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    S = Xn @ Xn.T
    _, si = np.unique(rec, return_inverse=True)
    _, li = np.unique(spk, return_inverse=True)
    excl = si[:, None] == si[None, :]
    same = (li[:, None] == li[None, :]) & (~excl)
    Sm = np.where(excl, -np.inf, S)

    order = np.argsort(-Sm, axis=1)
    ss = np.take_along_axis(same, order, axis=1)
    rel = same.sum(1); ok = rel > 0
    ranks = np.arange(1, S.shape[0] + 1)
    prec = np.cumsum(ss, axis=1) / ranks
    ap = (prec * ss).sum(1) / np.maximum(rel, 1)

    sc = S[~excl]; y = same[~excl]
    o = np.argsort(-sc); ys = y[o].astype(float)
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    frr = 1 - tp / max(tp[-1], 1); far = fp / max(fp[-1], 1)
    k = int(np.argmin(np.abs(frr - far)))
    return dict(eer=float((frr[k] + far[k]) / 2),
                rank1=float(ss[ok, 0].mean()),
                rank5=float(ss[ok, :5].any(1).mean()),
                mAP=float(ap[ok].mean()))


def _session_convergence(X, spk, rec):
    """Mean cosine similarity for the three pair types. The design claims
    same-session pairs should be much more similar than different-session ones."""
    Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    S = Xn @ Xn.T
    _, si = np.unique(rec, return_inverse=True)
    _, li = np.unique(spk, return_inverse=True)
    off = ~np.eye(len(S), dtype=bool)
    same_sess = (si[:, None] == si[None, :]) & off
    same_spk_diff_sess = (li[:, None] == li[None, :]) & (si[:, None] != si[None, :])
    diff_spk = li[:, None] != li[None, :]
    f = lambda m: float(S[m].mean()) if m.any() else float("nan")
    return f(same_sess), f(same_spk_diff_sess), f(diff_spk)


rows = []
for cfg in VARIANTS:
    p = os.path.join(OUT_DIR, f"fp_{cfg['name']}.npz")
    if not os.path.exists(p):
        print(f"[skip] {cfg['name']}: {p} missing"); continue
    d = np.load(p, allow_pickle=True)
    spk, rec = d["person_ids"], np.char.add(np.char.add(
        d["person_ids"], "/"), d["session_ids"])
    ha = np.asarray(d["hidden_activity"], dtype=np.float32)

    X = _flat(d, COMPONENTS)
    pc5, pc20 = _pc_frac(X)
    hpc5, _ = _pc_frac(np.asarray(d["hidden_activity"], dtype=np.float32))
    ipc5, _ = _pc_frac(np.asarray(d["input_activity"], dtype=np.float32))
    m = _retrieval(X, spk, rec)
    m_iw = _retrieval(_flat(d, ("in_weights",)), spk, rec)
    m_ia = _retrieval(_flat(d, ("input_activity",)), spk, rec)
    ss, sd_, dd = _session_convergence(X, spk, rec)

    rows.append(dict(
        name=cfg["name"], n=len(spk), dim=X.shape[1],
        eer=m["eer"], r1=m["rank1"], mAP=m["mAP"],
        eer_iw=m_iw["eer"], eer_ia=m_ia["eer"],
        pc5=pc5, pc20=pc20, ipc5=ipc5, hpc5=hpc5,
        dead=float((ha == 0).mean()), dead_all=float((ha.max(0) == 0).mean()),
        fisher=_fisher(X, spk), ss=ss, sd=sd_, dd=dd,
        sec=float(d["sec_per_wav"]),
    ))

print("MAIN COMPARISON  (all variants on identical clips — paired)\n")
print(f"{'variant':>12} {'EER':>7} {'R@1':>7} {'mAP':>7} | {'EER iw':>7} {'EER ia':>7} | "
      f"{'PC1-5':>6} {'PC1-20':>7} {'hid PC5':>8} | {'dead':>6} {'F':>5} | {'s/wav':>6}")
print("-" * 108)
for r in rows:
    print(f"{r['name']:>12} {r['eer']:>7.4f} {r['r1']:>7.4f} {r['mAP']:>7.4f} | "
          f"{r['eer_iw']:>7.4f} {r['eer_ia']:>7.4f} | {r['pc5']:>6.3f} {r['pc20']:>7.3f} "
          f"{r['hpc5']:>8.3f} | {r['dead']*100:>5.1f}% {r['fisher']:>5.2f} | {r['sec']:>6.1f}")
print("-" * 108)
print("EER/R@1/mAP  : cross-session speaker retrieval on the fingerprint alone. LOWER EER better.")
print("EER iw / ia  : same, using only in_weights / only input_activity.")
print("PC1-5/PC1-20 : share of variance in the top components = redundancy. LOWER IS BETTER —")
print("               this is the quantity change #1 (narrower receptive field) targets.")
print("hid PC5      : same for hidden_activity alone; baseline had 0.856 vs 0.684 for its own")
print("               input, i.e. the hidden layer was MORE redundant than what fed it.")
print("dead         : fraction of (clip, hidden unit) pairs with zero spikes. LOWER better.")
print("F            : between-speaker / within-speaker variance. 1.0 = no speaker structure.")

print("\n\nSESSION CONVERGENCE  (the design's own claim, measurable here for the first time)\n")
print(f"{'variant':>12} {'same session':>13} {'same spk, diff sess':>21} {'diff speaker':>13} "
      f"{'session gap':>12}")
print("-" * 78)
for r in rows:
    print(f"{r['name']:>12} {r['ss']:>13.4f} {r['sd']:>21.4f} {r['dd']:>13.4f} "
          f"{r['ss']-r['sd']:>12.4f}")
print("-" * 78)
print("Mean cosine similarity between fingerprints, by pair type.")
print("The claim is that utterances from the SAME session converge to nearly the same")
print("fingerprint. That holds if `same session` is clearly above `same spk, diff sess`.")
print("A large session gap also means session-level AVERAGING is available as free denoising.")


## How to read the result

### Decision gate (from `FINGERPRINT_IMPROVE.md` §5)

Judge every variant against this sweep's own `baseline` row, not against the 0.345 in
`RESULTS.md` — that number came from a supervised readout on the full corpus and is not
comparable to these unsupervised numbers.

- **A variant clearly beats `baseline` on EER *and* lowers `PC1-5`** → the diagnosis was
  right. Regenerate one full shard with that setting, then re-run `ecapa_film_snn.ipynb`
  cell 14 Part B and check whether fingerprint-alone EER drops below **0.28**. If it does,
  re-run the fusion test (cell 15), then a short FiLM run.
- **Nothing moves** → the weakness is not fan-in, dead units, or exposure length, and the
  honest call is to stop pursuing SNN-fingerprint conditioning for ECAPA.

### Reading the individual columns

- `ep4` vs `baseline` is purely a **cost** question. If EER and PC1-5 are close, you have just
  made the whole pipeline 4x cheaper (32s → 8s of simulated time per wav, over ~167k wavs)
  and every later experiment becomes affordable.
- `ep4_r5` is the **real** test. Watch `PC1-5` and `hid PC5` first — those are what the change
  is designed to move. EER should follow if the mechanism is right.
- Watch `dead` when reading `ep4_r5`: a narrower receptive field means a neuron hears only 11
  channels, so it can go silent more easily. If `dead` goes **up** in `ep4_r5` and `ep4_r5_vth`
  fixes it, the two changes need each other and should be adopted together.
- `F` near 1.0 means no speaker structure at all, whatever the EER says.

### If the session-convergence table shows a large gap

Then the design's premise holds and the next move is session-averaged fingerprints, which is
cheaper and more likely to help than anything in this sweep. If the gap is small, that premise
needs revisiting before more effort goes into the SNN.

### Caveat

Small sample by design (60 speakers). Fine for **ranking** variants, since they share the exact
same clips, but the absolute EER values are noisy — do not quote them as corpus-level results.